In [3]:
# %conda install --quiet numpy numba
import numpy as np
from numpy import typing as npt
import numba

2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: ...working... done

# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


In [6]:
Char = np.uint8
Enc = npt.NDArray[Char]
NEnc = numba.types.uint8[:]
NTable = numba.types.uint8[:, :, :]

CHARS = "abcdefghijklmnopqrstuvwxyz0123456789 "
C = len(CHARS)


def encode(s: str):
    return np.array([CHARS.index(c) for c in s], dtype=Char)


In [7]:
Con = np.uint16
NCons = numba.types.uint16[:, :, :]
NHash = numba.types.uint16[:]

MQ = np.iinfo(Con).max + 1
M = 100000
Q = MQ / M

N = 14
P0 = np.array([11, 17, 7, 5], np.uint16).reshape(-1, 1)
P1 = np.array([29, 31, 17, 13], np.uint16).reshape(-1, 1)
P2 = np.array([53, 67, 103, 47], np.uint16).reshape(-1, 1)
P3 = np.array([52, 12, 24, 30], np.uint16).reshape(-1, 1)
P4 = np.array([0, 90, 0, 90], np.uint16).reshape(-1, 1)
idx = np.arange(N, dtype=np.uint16)
_A = np.outer(P0, idx + 1) % P1 + P2
_B = _A * P3 + P4
A = _A[:, :, np.newaxis]
B = _B[:, :, np.newaxis]


@numba.jit(NCons(NTable), fastmath=True)
def contribution(e: Enc):
    angle = A * e + B
    sin = 5 * np.sin(np.radians(angle))
    return ((sin - np.floor(sin)) * MQ).astype(Con)


etable = np.outer(np.ones(N, dtype=Char), np.arange(C, dtype=Char))
E = etable[np.newaxis, :, :]
CTABLE = contribution(E)


@numba.jit(NHash(NEnc))
def hashify(e: Enc):
    result = np.zeros(4, dtype=Con)
    for i in numba.prange(N):
        result += CTABLE[:, i, e[i]]
    return result


enc = encode("hello worldabc")  # [7,4,11,11,14,36,22,14,17,11,3,0,1,2]
h = hashify(enc)

np.testing.assert_array_almost_equal(
    hashify(encode("hello worldabc")),
    np.array([48723, 83530, 72842, 86833]) * Q,
    decimal=-1,
)
np.testing.assert_array_almost_equal(
    hashify(encode("hfrxdl9t6oj1uo")),
    np.array([90848, 78941, 97533, 76144]) * Q,
    decimal=-1,
)
np.testing.assert_array_almost_equal(
    hashify(encode("jg 4nk8qjusyq ")),
    np.array([42749, 51443, 58210, 55835]) * Q,
    decimal=-1,
)
np.testing.assert_array_almost_equal(
    hashify(encode("v6hidyaxbg2sz6")),
    np.array([82976, 64712, 86038, 1066]) * Q,
    decimal=-1,
)
hashify(encode("hello worldabc")), np.array([48723, 83530, 72842, 86833]) * Q


(array([31925, 54735, 47729, 56900], dtype=uint16),
 array([31931.10528, 54742.2208 , 47737.73312, 56906.87488]))

In [8]:
%timeit encode("hello worldabc")
%timeit hashify(enc)
# 3.46 µs ± 53.6 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)
# 1.17 µs ± 318 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)

4 μs ± 138 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
1.19 μs ± 27.8 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [15]:
import numpy as np
from numpy import typing as npt
import numba

np.random.seed(0)

Key = np.uint64
NKey = numba.types.uint64
Keys = npt.NDArray[Key]
NKeys = NKey[:]

size = 37 ** 6
# size = 1_000_000
print(f"Generating {size} unique random keys...")
keys = np.random.randint(0, np.iinfo(Key).max, size, dtype=Key)


Generating 2565726409 unique random keys...


In [ ]:
segment_length = 1 << int(np.log(size) / np.log(3.33) + 2.25)
segment_length_mask = segment_length - 1

total_segments = (int(size * 1.125) + segment_length - 1) // segment_length
arity = 3
segment_count = total_segments - (arity - 1)
segment_count_length = segment_count * segment_length


@numba.jit(numba.types.containers.UniTuple(numba.types.uint32, arity)(NKey))
def _get_hashes(key: Key):
    a_low = key & 0xFFFFFFFF
    a_high = key >> 32
    b_low = segment_count_length & 0xFFFFFFFF
    b_high = segment_count_length >> 32
    p00 = a_low * b_low
    p01 = a_low * b_high
    p10 = a_high * b_low
    p11 = a_high * b_high
    middle = p10 + (p00 >> 32) + (p01 & 0xFFFFFFFF)
    h0 = p11 + (middle >> 32) + (p01 >> 32)
    h1 = h0 + segment_length
    h2 = h1 + segment_length
    h1 ^= np.uint32((key >> 18) & segment_length_mask)
    h2 ^= np.uint32(key & segment_length_mask)
    return h0, h1, h2


@numba.jit(NKeys(NKeys))
def populate(keys: Keys):
    block_bits = 1
    while (1 << block_bits) < segment_count:
        block_bits += 1
    block = 1 << block_bits
    start_pos = np.zeros(block, dtype=np.uint32)

    for i in range(block):
        start_pos[i] = np.uint32((i * size) >> block_bits)

    reverse_order = np.zeros(size + 1, dtype=Key)
    reverse_order.fill(0)
    reverse_order[size] = 1
    mask_block = Key(block - 1)
    for h in keys:
        segment_index = h >> (64 - block_bits)
        while reverse_order[start_pos[segment_index]] != 0:
            segment_index = (segment_index + Key(1)) & mask_block
        reverse_order[start_pos[segment_index]] = h
        start_pos[segment_index] += 1

    capacity = total_segments * segment_length
    t2count = np.zeros(capacity, dtype=np.uint8)
    t2hash = np.zeros(capacity, dtype=Key)
    for i in range(size):
        hash_val = reverse_order[i]
        h0, h1, h2 = _get_hashes(hash_val)
        t2count[h0] += 4
        t2hash[h0] ^= hash_val
        t2count[h1] += 4
        t2count[h1] ^= 1
        t2hash[h1] ^= hash_val
        t2count[h2] += 4
        t2count[h2] ^= 2
        t2hash[h2] ^= hash_val

    alone = np.zeros(capacity, dtype=np.uint32)
    q_size = 0
    for i in range(capacity):
        if (t2count[i] >> 2) == 1:
            alone[q_size] = i
            q_size += 1

    reverse_h = np.zeros(size, dtype=np.uint8)
    stack_size = 0
    while q_size > 0:
        q_size -= 1
        index = alone[q_size]
        if (t2count[index] >> 2) == 1:
            hash_val = t2hash[index]
            found = t2count[index] & 3
            reverse_h[stack_size] = found
            reverse_order[stack_size] = hash_val
            stack_size += 1
            h_all = _get_hashes(hash_val)

            other_index1 = h_all[(found + 1) % 3]
            if (t2count[other_index1] >> 2) == 2:
                alone[q_size] = other_index1
                q_size += 1
            t2count[other_index1] -= 4
            t2count[other_index1] ^= (found + 1) % 3
            t2hash[other_index1] ^= hash_val

            other_index2 = h_all[(found + 2) % 3]
            if (t2count[other_index2] >> 2) == 2:
                alone[q_size] = other_index2
                q_size += 1
            t2count[other_index2] -= 4
            t2count[other_index2] ^= (found + 2) % 3
            t2hash[other_index2] ^= hash_val

    fingerprints = np.zeros(capacity, dtype=Key)
    for i in range(size - 1, -1, -1):
        hash_val = reverse_order[i]
        found = reverse_h[i]
        h_all = _get_hashes(hash_val)
        fingerprints[h_all[found]] = (
            hash_val
            ^ fingerprints[h_all[(found + 1) % 3]]
            ^ fingerprints[h_all[(found + 2) % 3]]
        )
    return fingerprints


print("Populating filter...")
fingerprints = populate(keys)
print("Filter populated successfully.")
np.save('fingerprints.npy', fingerprints)

Populating filter...
Filter populated successfully.


In [ ]:
%timeit populate(keys)

In [ ]:
fingerprints = np.load("fingerprints.npy")


@numba.vectorize([numba.types.boolean(NKey)])
def contain(key: Key):
    h0, h1, h2 = _get_hashes(key)
    return (key ^ fingerprints[h0] ^ fingerprints[h1] ^ fingerprints[h2]) == 0


# Verification
found_count = contain(keys).sum()
print(f"Found {found_count}/{size} of the original keys.")

# False positive test
num_test_keys = 100000
test_keys = np.random.randint(0, np.iinfo(Key).max, size=num_test_keys, dtype=Key)

false_positives = contain(test_keys).sum()

print(f"False positive rate: {false_positives / num_test_keys:.4f}")

assert found_count == size
assert false_positives / num_test_keys < 0.01

In [14]:
%timeit contain(keys)

20.5 ms ± 905 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
import numpy as np
import numba

L = 5

np.random.seed(42)
target = np.random.randint(0, C, L)


@numba.jit
def check(a: np.ndarray):
    return np.array_equal(a, target)


@numba.jit(parallel=True)
def search():
    done = False
    res = np.zeros(L, dtype=Con)
    for i in numba.prange(C):
        if done:
            continue
        enc = np.full(L, C - 1, dtype=Con)
        enc[0] = i
        while True:
            if check(enc):
                done = True
                for k in range(N):
                    res[k] = enc[k]
                break
            j = L - 1
            while j > 0:
                if enc[j]:
                    enc[j] -= 1
                    break
                else:
                    enc[j] = C - 1
                    j -= 1
            else:
                break
    return res


# search()


In [ ]:
%timeit search()